# Phase 1: EDA & Data Cleaning

**Author:** [Your Name]  
**Last run:** [Date]  
**Goal:** Produce a clean, well-understood dataset ready for modeling.  
**Output:** `../../data/processed/stroke_cleaned.csv`

### Checklist
- [ ] Load raw data; sanity checks
- [ ] EDA: distributions, class balance, correlations
- [ ] Handle missing BMI
- [ ] Drop `id`; handle rare categories
- [ ] Encode categoricals; scale numerics
- [ ] Save processed dataset

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Put repo root on sys.path so `from src...` imports work from this notebook
sys.path.append(os.path.abspath(os.path.join('..', '..')))

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

RAW_PATH = os.path.join('..', '..', 'data', 'raw', 'healthcare-dataset-stroke-data.csv')
PROCESSED_PATH = os.path.join('..', '..', 'data', 'processed', 'stroke_cleaned.csv')
FIGURES_PATH = os.path.join('..', '..', 'results', 'figures')

## 1. Load & Sanity Check

In [ ]:
# Auto-download the raw CSV from Kaggle if it isn't already in data/raw/.
# Needs Kaggle credentials once per machine (see docs/user_manual.md).
# If you'd rather download manually, skip this cell — just put the CSV at RAW_PATH.
if not os.path.exists(RAW_PATH):
    from src.data import ensure_dataset
    RAW_PATH = ensure_dataset()
print(f'Using raw data: {RAW_PATH}')

In [ ]:
df = pd.read_csv(RAW_PATH)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())

## 2. Class Distribution

In [ ]:
print(df['stroke'].value_counts())
print(f"\nPositive class rate: {df['stroke'].mean():.2%}")

fig, ax = plt.subplots()
df['stroke'].value_counts().plot(kind='bar', ax=ax, color=['steelblue', 'tomato'])
ax.set_title('Class Distribution: Stroke vs No Stroke')
ax.set_xticklabels(['No Stroke (0)', 'Stroke (1)'], rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'p1_class_distribution.png'), dpi=150)
plt.show()

## 3. Feature Distributions

In [ ]:
numeric_cols = ['age', 'avg_glucose_level', 'bmi']
df[numeric_cols].hist(bins=30, figsize=(12, 4))
plt.suptitle('Numeric Feature Distributions')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'p1_numeric_distributions.png'), dpi=150)
plt.show()

In [ ]:
cat_cols = ['gender', 'hypertension', 'heart_disease', 'ever_married',
            'work_type', 'Residence_type', 'smoking_status']
for col in cat_cols:
    print(df[col].value_counts(), '\n')

## 4. Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[numeric_cols + ['hypertension', 'heart_disease', 'stroke']].corr(),
            annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Correlation Heatmap')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'p1_correlation_heatmap.png'), dpi=150)
plt.show()

## 5. Data Cleaning

In [ ]:
df_clean = df.copy()

# Drop id
df_clean.drop(columns=['id'], inplace=True)

# Handle gender = 'Other' (1 record) — drop or merge
print(f"Records with gender='Other': {(df_clean['gender'] == 'Other').sum()}")
df_clean = df_clean[df_clean['gender'] != 'Other']

# Impute BMI with median
bmi_median = df_clean['bmi'].median()
df_clean['bmi'].fillna(bmi_median, inplace=True)
print(f'BMI missing after imputation: {df_clean["bmi"].isnull().sum()}')

# Smoking status 'Unknown' stays as its own category
print(df_clean['smoking_status'].value_counts())

## 6. Encoding & Scaling

In [ ]:
df_encoded = pd.get_dummies(df_clean,
                             columns=['gender', 'ever_married', 'work_type',
                                      'Residence_type', 'smoking_status'],
                             drop_first=True)
print(f'Shape after encoding: {df_encoded.shape}')
df_encoded.head()

## 7. Save Processed Dataset

In [ ]:
os.makedirs(os.path.dirname(PROCESSED_PATH), exist_ok=True)
df_encoded.to_csv(PROCESSED_PATH, index=False)
print(f'Saved to {PROCESSED_PATH}')